# 05 · Análisis descriptivo de las prórrogas

**Qué hace:** calcula la tasa de prórroga (porcentaje de contratos con días añadidos) por
tipo de contrato, modalidad, entidad, departamento, mes y año, con tablas y gráficos.

**Cómo:** todo se calcula con agregaciones de Spark (`groupBy` + `agg`). Lo que se trae a
Python para mostrar o graficar es solo el **resultado agregado** (unas decenas de filas),
nunca la tabla completa: `toPandas()` sobre `gold` cargaría cientos de miles de filas en
la memoria del Codespace.

Solo se usan años con etiqueta completa (2025-2026 aún pueden recibir prórrogas y
sesgarían la tasa hacia abajo).

**Resultados:** PENDIENTE (se anotan en `docs/informe.md` después de ejecutar).

## Configuración: importar `src/secop` y crear la sesión de Spark

1. Buscamos la carpeta del repositorio (la que contiene `src/secop`) subiendo desde la
   carpeta del notebook y añadimos `src` a `sys.path`, la lista de carpetas donde Python
   busca módulos. Así `import secop` funciona sin instalar nada.
2. `entorno.crear_spark()` crea una sesión de **Spark en modo local** con Delta Lake: el
   mismo motor que en un clúster, usando los núcleos del Codespace.

Si cambias código en `src/`, reinicia el kernel (botón *Restart*) para que Python lo
vuelva a cargar.

In [ ]:
import sys
from pathlib import Path


def buscar_raiz_repo() -> Path:
    """Devuelve la carpeta raíz del repo (la que contiene src/secop)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "src" / "secop").is_dir():
            return carpeta
    raise RuntimeError("No encontré src/secop: abre el notebook desde la carpeta del repositorio.")


RAIZ_REPO = buscar_raiz_repo()
if str(RAIZ_REPO / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ_REPO / "src"))

import secop
from secop import config, entorno

spark = entorno.crear_spark()
print("secop", secop.__version__, "| Spark", spark.version, "| datos en", config.raiz_datos())

## 0. Preparación
`MIN_CONTRATOS` descarta grupos diminutos cuya tasa no sería confiable: una entidad con 2
contratos y 1 prórroga tendría 50 %, y eso no significa nada.

In [ ]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from secop import graficos
from secop.descriptivo import tasa_por_grupo
from secop.modelado import solo_etiqueta_completa

MIN_CONTRATOS = int(entorno.parametro("MIN_CONTRATOS", "100"))
gold = solo_etiqueta_completa(entorno.leer_tabla(spark, config.TABLA_ORO))


def grafico_de_barras(df_agregado, columna, titulo, filas=15):
    """Trae a pandas SOLO las primeras filas del resultado agregado y las grafica."""
    pdf = df_agregado.limit(filas).toPandas()
    graficos.barras_tasa(pdf, columna, TASA_GLOBAL, titulo)
    plt.show()

## 1. Tasa global
Es la referencia para todo lo demás: un grupo "alto" es uno por encima de esta tasa.
También es la AUC-PR de un modelo que no sabe nada (ver notebook 06).

In [ ]:
global_ = gold.agg(
    F.count("*").alias("contratos"),
    F.sum("etiqueta").alias("prorrogados"),
    F.round(F.avg("etiqueta") * 100, 2).alias("tasa_prorroga_pct"),
)
entorno.mostrar(global_)
TASA_GLOBAL = float(global_.first()["tasa_prorroga_pct"] or 0)

## 2. Por año y mes de firma
¿Hay estacionalidad? Por ejemplo, contratos firmados a fin de año que se prorrogan para
terminar en la vigencia siguiente.

In [ ]:
entorno.mostrar(tasa_por_grupo(gold, ["anio_firma"]).orderBy("anio_firma"))

por_mes = (
    tasa_por_grupo(gold, ["anio_firma", "mes_firma"])
    .orderBy("anio_firma", "mes_firma")
    .withColumn("periodo", F.format_string("%d-%02d", "anio_firma", "mes_firma"))
)
graficos.linea_tasa(
    por_mes.toPandas(), "periodo", "Tasa de prórroga por mes de firma"
)  # ≤ 24 filas
plt.show()

## 3. Por tipo de contrato y modalidad

In [ ]:
por_tipo = tasa_por_grupo(gold, ["tipo_de_contrato"], MIN_CONTRATOS)
entorno.mostrar(por_tipo)
grafico_de_barras(por_tipo, "tipo_de_contrato", "Tasa de prórroga por tipo de contrato")

In [ ]:
por_modalidad = tasa_por_grupo(gold, ["modalidad_de_contratacion"], MIN_CONTRATOS)
entorno.mostrar(por_modalidad)
grafico_de_barras(por_modalidad, "modalidad_de_contratacion", "Tasa de prórroga por modalidad")

## 4. Por departamento y por entidad
Para mostrar el nombre de la entidad se une el resultado agregado (pequeño) con
`dim_entidad` usando broadcast.

In [ ]:
por_departamento = tasa_por_grupo(gold, ["departamento_entidad"], MIN_CONTRATOS)
entorno.mostrar(por_departamento, filas=40)
grafico_de_barras(
    por_departamento, "departamento_entidad", "Departamentos con mayor tasa de prórroga"
)

In [ ]:
entidades = entorno.leer_tabla(spark, config.TABLA_DIM_ENTIDAD).select(
    "codigo_entidad", "nombre_entidad", "orden_entidad"
)
por_entidad = (
    tasa_por_grupo(gold, ["codigo_entidad"], MIN_CONTRATOS)
    .join(F.broadcast(entidades), "codigo_entidad", "left")
    .select("nombre_entidad", "orden_entidad", "contratos", "prorrogados", "tasa_prorroga_pct")
    .orderBy(F.desc("tasa_prorroga_pct"), F.desc("contratos"))
)
entorno.mostrar(por_entidad, filas=30)
grafico_de_barras(por_entidad, "nombre_entidad", "Entidades con mayor tasa de prórroga")

## 5. Otras variables al firmar
Cláusula de prórroga, inicio antes de la firma y segmento UNSPSC.

In [ ]:
entorno.mostrar(tasa_por_grupo(gold, ["puede_ser_prorrogado"]))
entorno.mostrar(tasa_por_grupo(gold, ["inicio_antes_de_firma"]))
entorno.mostrar(tasa_por_grupo(gold, ["segmento_unspsc"], MIN_CONTRATOS), filas=30)

## 6. Diagnóstico de fuga: plazo según la etiqueta
Si `fecha_fin` ya incluye los días añadidos, los contratos prorrogados tendrán un plazo
(fin − inicio) mucho mayor que su duración declarada. Comparamos medianas por clase.
Una diferencia grande entre `plazo_dias` y `duracion_dias_declarada` **solo** en la clase 1
sería evidencia de fuga.

In [ ]:
entorno.mostrar(
    gold.groupBy("etiqueta").agg(
        F.count("*").alias("contratos"),
        F.percentile_approx("plazo_dias", 0.5).alias("mediana_plazo_dias"),
        F.percentile_approx("duracion_dias_declarada", 0.5).alias("mediana_duracion_declarada"),
        F.percentile_approx(F.col("plazo_dias") - F.col("duracion_dias_declarada"), 0.5).alias(
            "mediana_diferencia"
        ),
    )
)